In [1]:
import json
import os
import glob
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve
from catboost import CatBoostClassifier

# 📌 외부 파일에서 딥러닝 모델 클래스 임포트
from models import ChurnLSTM, ChurnCNN1D, ChurnTransformer


In [2]:
# ==========================================
# 🚀 1. 디렉토리 구조 설정 및 테스트셋 준비
# ==========================================
DATA_DIR = "./data/data/processed"
ML_MODEL_DIR = "./model/machine_learning"
DL_MODEL_DIR = "./model/deep_learning"
EVAL_DIR = "./test"
TEST_CSV_PATH = "./data/data/processed/test_20181101_v1.csv"  # 테스트용 정형 데이터 파일
TEST_NPY_PATH = "./data/data/processed/test.npy"

os.makedirs(EVAL_DIR, exist_ok=True)

print("=" * 50)
print("🚀 1. 테스트셋 데이터 로드 및 준비 (개별 평가 전용)")
print("=" * 50)

# 1. 테스트 정형 데이터 및 시퀀스 텐서 직접 로드
df_test = pd.read_csv(TEST_CSV_PATH)

if os.path.exists(TEST_NPY_PATH):
    X_seq_test = np.load(TEST_NPY_PATH)
else:
    raise FileNotFoundError(f"'{TEST_NPY_PATH}' 파일을 찾을 수 없습니다. 시퀀스 변환 코드를 먼저 실행해 주세요.")

# 타겟 컬럼 지정
target_col = "churn" if "churn" in df_test.columns else ("is_churn" if "is_churn" in df_test.columns else df_test.columns[-1])
y_test = df_test[target_col].values
y_true = np.array(y_test).ravel()

# 🛠️ 불필요한 컬럼 제거 및 피처 추출
drop_cols = [target_col, "user_id", "userId", "page_sequence", "session_item_sequence", "padded_sequence"]
feature_cols = [col for col in df_test.columns if col not in drop_cols]
X_tab_test = df_test[feature_cols].copy()

# 범주형 컬럼 인코딩
for col in ["gender", "level", "last_page", "level_last"]:
    if col in X_tab_test.columns:
        X_tab_test[col] = X_tab_test[col].astype("category").cat.codes

# 객체 타입 컬럼 숫자형 변환 및 결측치 처리
for col in X_tab_test.columns:
    if X_tab_test[col].dtype == "object":
        X_tab_test[col] = pd.to_numeric(X_tab_test[col], errors="coerce").fillna(0.0)

print(f"✅ X_tab_test 정비 완료! 최종 shape: {X_tab_test.shape} (피처 수: {X_tab_test.shape[1]}개)")
print(f"✅ X_seq_test 텐서 shape: {X_seq_test.shape}")

# 딥러닝(DL) 테스트 데이터로더 구성
test_dataset = TensorDataset(torch.tensor(X_seq_test, dtype=torch.long), torch.tensor(y_test, dtype=torch.float32))
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

🚀 1. 테스트셋 데이터 로드 및 준비 (개별 평가 전용)
✅ X_tab_test 정비 완료! 최종 shape: (2721, 14) (피처 수: 14개)
✅ X_seq_test 텐서 shape: (2721, 50)


In [3]:
# ==========================================
# 🚀 2. 모델 디렉토리 스캔 및 예측값 수집 (안전한 분기 처리)
# ==========================================
print("\n" + "=" * 50)
print("🚀 2. 저장된 모델 스캔 및 예측 수행")
print("=" * 50)

# 2.1 머신러닝 모델 예측값 수집 (.pkl 및 .cbm 파일 동시 지원)
ml_predictions = {}
ml_model_files = glob.glob(os.path.join(ML_MODEL_DIR, "*.pkl")) + glob.glob(os.path.join(ML_MODEL_DIR, "*.cbm"))

if not ml_model_files:
    print(f"⚠️ 경고: {ML_MODEL_DIR} 경로에 저장된 머신러닝 모델(.pkl, .cbm)이 없습니다.")

for model_path in ml_model_files:
    model_name = os.path.basename(model_path)
    
    if model_name.endswith(".pkl"):
        model = joblib.load(model_path)
    elif model_name.endswith(".cbm"):
        model = CatBoostClassifier()
        model.load_model(model_path)
    else:
        raise ValueError(f"지원하지 않는 머신러닝 모델 형식입니다: {model_name}")
        
    # 💡 [핵심 해결] 모델 종류에 따른 안전한 예측 수행
    try:
        # CatBoost 모델이거나 n_features_in_ 속성이 있는 경우
        if isinstance(model, CatBoostClassifier) or hasattr(model, "n_features_in_"):
            n_feat = getattr(model, "n_features_in_", X_tab_test.shape[1])
            # 학습 피처 개수에 맞게 슬라이싱하여 전달
            preds = model.predict_proba(X_tab_test.iloc[:, :n_feat])[:, 1]
        else:
            preds = model.predict_proba(X_tab_test)[:, 1]
    except Exception as e:
        # LightGBM 등에서 피처 검증 에러가 날 경우 validate_features=False 적용
        try:
            preds = model.predict_proba(X_tab_test, validate_features=False)[:, 1]
        except Exception as e2:
            # 마지막 안전장치: 강제로 numpy array로 변환 후 예측
            preds = model.predict_proba(X_tab_test.values)[:, 1]
            
    ml_predictions[model_name] = preds
    print(f"📥 [ML 로드 완료] {model_name} (Test AUC: {roc_auc_score(y_true, preds):.4f})")

# 2.2 딥러닝 모델 예측값 수집
dl_predictions = {}
dl_model_files = glob.glob(os.path.join(DL_MODEL_DIR, "*.pth"))
if not dl_model_files:
    print(f"⚠️ 경고: {DL_MODEL_DIR} 경로에 저장된 딥러닝 모델 가중치(.pth)가 없습니다.")

for model_path in dl_model_files:
    model_name = os.path.basename(model_path)
    
    if "lstm" in model_name.lower():
        dl_model = ChurnLSTM(vocab_size=50, embedding_dim=32, hidden_dim=64).to(device)
    elif "cnn" in model_name.lower():
        dl_model = ChurnCNN1D(vocab_size=50, embedding_dim=32, num_filters=64).to(device)
    elif "transformer" in model_name.lower():
        dl_model = ChurnTransformer(vocab_size=50, embedding_dim=32, num_heads=4, hidden_dim=64, num_layers=2, dropout=0.1).to(device)
    else:
        raise ValueError(f"지원하지 않는 딥러닝 모델 파일명입니다: {model_name}")
        
    dl_model.load_state_dict(torch.load(model_path, map_location=device))
    dl_model.eval()
    
    dl_preds_list = []
    with torch.no_grad():
        for seqs, _ in test_loader:
            outputs = torch.sigmoid(dl_model(seqs.to(device)))
            dl_preds_list.extend(outputs.cpu().numpy())
    
    preds = np.array(dl_preds_list).ravel()
    dl_predictions[model_name] = preds
    print(f"📥 [DL 로드 완료] {model_name} (Test AUC: {roc_auc_score(y_true, preds):.4f})")


🚀 2. 저장된 모델 스캔 및 예측 수행
📥 [ML 로드 완료] ab_model.pkl (Test AUC: 0.7292)
📥 [ML 로드 완료] dt_model.pkl (Test AUC: 0.6880)
📥 [ML 로드 완료] et_model.pkl (Test AUC: 0.7264)
📥 [ML 로드 완료] hgb_model.pkl (Test AUC: 0.7216)
📥 [ML 로드 완료] knn_model.pkl (Test AUC: 0.5990)
📥 [ML 로드 완료] lgb_model.pkl (Test AUC: 0.7373)
📥 [ML 로드 완료] lr_model.pkl (Test AUC: 0.7226)
📥 [ML 로드 완료] mlp_model.pkl (Test AUC: 0.5882)
📥 [ML 로드 완료] nb_model.pkl (Test AUC: 0.6966)
📥 [ML 로드 완료] rf_model.pkl (Test AUC: 0.7259)
📥 [ML 로드 완료] svc_model.pkl (Test AUC: 0.4927)
📥 [ML 로드 완료] xgb_model.pkl (Test AUC: 0.7331)
📥 [ML 로드 완료] catboost_model.cbm (Test AUC: 0.7337)
📥 [DL 로드 완료] cnn1d_model_weights.pth (Test AUC: 0.8453)
📥 [DL 로드 완료] lstm_model_weights.pth (Test AUC: 0.8685)
📥 [DL 로드 완료] transformer_model_weights.pth (Test AUC: 0.8631)


In [4]:
# ==========================================
# 🚀 4. 모든 ML x DL 조합 중 최고 성능 탐색
# ==========================================
if not ml_predictions or not dl_predictions:
    raise ValueError("앙상블을 수행할 ML 모델 또는 DL 모델 파일이 부족합니다. 디렉토리를 확인해주세요.")

best_overall_auc = 0
best_combination = None
best_hybrid_preds = None
best_weights = (0.5, 0.5)

for ml_name, ml_preds in ml_predictions.items():
    for dl_name, dl_preds in dl_predictions.items():
        for w in np.linspace(0.5, 0.95, 10):
            temp_preds = (w * ml_preds) + ((1.0 - w) * dl_preds)
            score = roc_auc_score(y_true, temp_preds)
            if score > best_overall_auc:
                best_overall_auc = score
                best_combination = (ml_name, dl_name)
                best_hybrid_preds = temp_preds
                best_weights = (w, 1.0 - w)

print("\n" + "★" * 50)
print(f"🏆 최적 하이브리드 조합 발견!")
print(f" - 머신러닝 모델: {best_combination[0]} (가중치: {best_weights[0]:.2f})")
print(f" - 딥러닝 모델: {best_combination[1]} (가중치: {best_weights[1]:.2f})")
print(f" - 🏆 최종 하이브리드 Test ROC-AUC Score: {best_overall_auc:.4f}")
print("★" * 50)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🏆 최적 하이브리드 조합 발견!
 - 머신러닝 모델: ab_model.pkl (가중치: 0.60)
 - 딥러닝 모델: lstm_model_weights.pth (가중치: 0.40)
 - 🏆 최종 하이브리드 Test ROC-AUC Score: 0.8994
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


In [5]:
# ==========================================
# 🚀 5. 최적 임계값 적용 및 평가 리포트 출력
# ==========================================
precisions, recalls, thresholds = precision_recall_curve(y_true, best_hybrid_preds)

p = precisions[:-1]
r = recalls[:-1]
t = thresholds

# 2. F_beta Score 계산 (beta = 2로 설정하여 재현율에 가중치 부여)
beta = 2.0
f_beta_scores = (
    (1 + beta**2) * (p * r) / ((beta**2 * p) + r + 1e-10)
)

# 3. F_beta가 가장 높은 최적의 임계값 선택
best_idx = np.argmax(f_beta_scores)
best_threshold = t[best_idx]

print(f"\n📌 재현율 우선 최적 임계값 (F2-Score 기준): {best_threshold:.4f}\n")
optimal_preds = (best_hybrid_preds >= best_threshold).astype(int)
print(classification_report(y_true, optimal_preds))

# 결과 설정 및 예측값 저장 (./test 디렉토리)
hybrid_config = {
    "best_ml_model": best_combination[0],
    "best_dl_model": best_combination[1],
    "ml_weight": float(best_weights[0]),
    "dl_weight": float(best_weights[1]),
    "best_auc": float(best_overall_auc),
    "best_threshold": float(best_threshold)
}
with open(os.path.join(EVAL_DIR, "best_hybrid_config.json"), "w", encoding="utf-8") as f:
    json.dump(hybrid_config, f, ensure_ascii=False, indent=4)

np.save(os.path.join(EVAL_DIR, "best_hybrid_preds.npy"), best_hybrid_preds)
print("\n모든 추론, 최적화 및 결과 저장 과정이 ./test 디렉토리에 완료되었습니다!")


📌 재현율 우선 최적 임계값 (F2-Score 기준): 0.3982

              precision    recall  f1-score   support

           0       0.99      0.80      0.88      2566
           1       0.21      0.86      0.33       155

    accuracy                           0.80      2721
   macro avg       0.60      0.83      0.61      2721
weighted avg       0.94      0.80      0.85      2721


모든 추론, 최적화 및 결과 저장 과정이 ./test 디렉토리에 완료되었습니다!
